# Pipelyt Sandbox — Text + Image Generation Pipeline

Self-contained playground mirroring the live Pipelyt pipeline. Lets you iterate on prompts in Colab before touching production code.

**Pipeline stages reproduced in this notebook:**
1. Brief Guard (regex)
2. Refiner Agent (Gemini 2.5 Flash, no tools)
3. Cultural Calendar (Gemini 2.5 Flash + google_search, cached per session)
4. Researcher (Gemini 2.5 Flash + google_search, always grounded)
5. Copywriter (Gemini 2.5 Flash, no tools, v3 free-style prompt)
6. Image Agent v4 (Gemini image model, 5 variants)

**Inputs you control:**
- `CAMPAIGN_BRIEF` — your raw brief
- `BRAND_DNA_MODE` — `'none'` or `'spenzo'`
- `PLATFORMS` — subset of `["linkedin", "twitter", "facebook", "instagram"]`
- `ASPECT_RATIO` — `'1:1'` / `'9:16'` / `'16:9'` / `'4:5'`
- `IMAGE_MODEL` — defaults to `gemini-3.1-flash-image`, auto-falls back to `gemini-2.5-flash-image` if 3.1 unavailable

Run cells top-to-bottom. Each agent prints its output so you can inspect at every stage.

## 0 — Install + Imports

In [ ]:
!pip install -q google-genai pillow requests

In [ ]:
import os, re, json, time, uuid, base64, logging
from io import BytesIO
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass
from IPython.display import Image as IPyImage, display, Markdown

from google import genai
from google.genai import types
from PIL import Image
import requests

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
log = logging.getLogger('pipelyt')

## 1 — Gemini API key

Get one at https://aistudio.google.com/apikey . Paste when prompted (no echo).

In [ ]:
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') or getpass('GEMINI_API_KEY: ').strip()
client = genai.Client(api_key=GEMINI_API_KEY)
print('Client ready.')

## 2 — Embedded Business DNA (Spenzo AI + NeuZenAI parent)

Pulled from the contact@neuzenai.com account so you can iterate without DB access.

In [ ]:
SPENZO_DNA = {
  'product_name': 'Spenzo AI',
  'tagline': 'AI Powered Marketing Intelligence Platform',
  'overview': '''CORE OFFERINGS & MASTER VALUE PROPOSITION
Spenzo AI provides Unified Media Performance Intelligence, serving as an advanced Marketing Mix Modeling (MMM) intelligence platform. Its core offering is to transform the complex task of measuring marketing impact into a measurable and actionable process. The platform empowers modern growth teams to accurately measure campaign impact, forecast future outcomes, optimize marketing spend efficiently, and scale their operations with confidence using AI agents. The master value proposition is to maximize revenue by providing clear, decision-ready intelligence, automating data pipelines, and enabling strategic budget planning through conversational AI and dynamic insights.

EXPLICIT LIST OF ALL PRODUCTS & SERVICES
1. Spenzo AI Platform: The overarching intelligence platform for marketing mix modeling.
2. Integrations: A service providing one-click connectors to various data sources and ad platforms, ensuring auto-sync of spend and conversion data.
3. Dynamic Insights: A service offering a strategic insight dashboard to turn channel data into actionable intelligence, including performance tracking and market dynamics comparison.
4. Spenzo AI (Conversational Intelligence & Action): An AI-powered feature that allows users to generate insights, build charts, and train/adjust MMM models using simple text commands.
5. Budget Planning: A service for simulating budget allocations, enforcing constraints, and identifying high-return budget mixes.

SPECIFIC USE CASES AND CAPABILITIES
- MEASURING IMPACT: Quantifying the effectiveness and ROI of marketing campaigns and channels.
- FORECASTING OUTCOMES: Predicting future marketing performance and business results.
- OPTIMIZING SPEND: Strategically allocating marketing budgets across channels to maximise ROAS.
- DATA CONNECTION: Connectors for AWS S3, Google Cloud Storage, BigQuery, Azure Blob, Databricks, Snowflake, Funnel, HubSpot, Google Ads, Meta Ads, LinkedIn Ads, TikTok Ads.
- CONVERSATIONAL AI: Natural-language access to insights, charts, and MMM training.
- BUDGET SIMULATION: Fixed-budget and target-ROI optimisation strategies.
- TRANSPARENT MMM: Marketing Mix Modeling with clear, understandable results.

TARGET AUDIENCE AND INDUSTRIES
TARGET AUDIENCE:
- Modern growth teams.
- Marketing teams and professionals responsible for optimizing marketing spend and performance.
- Businesses managing significant annual marketing spend ($10M+).

INDUSTRIES:
Spenzo AI is designed for any industry that engages in marketing activities and seeks to optimize its media spend. Real customers include ENERPARC, Kavya, Ratnadeep, Repos Energy, Solar Square, Tata 1mg, Berlin, Tide — broad applicability across energy, e-commerce, consumer goods.''',
  'brand_values': ['Measurement', 'Optimization', 'Intelligence', 'Confidence', 'Efficiency', 'Transparency', 'Automation', 'Data-driven'],
  'brand_tone': ['Authoritative', 'Confident', 'Intelligent', 'Practical', 'Empowering', 'Direct'],
  'brand_aesthetic': ['Modern', 'Clean', 'Professional', 'Data-focused', 'Intuitive', 'Vibrant'],
  'fonts': ['Inter'],
  'colors': {'accent': '#E0E0E0', 'primary': '#FF5722', 'secondary': '#212B36', 'background': '#FFFFFF'},
  'url': 'spenzo.io',
  'logo_url': 'https://www.spenzo.ai/flux.png',
}

NEUZEN_COMPANY_DNA = {
  'company_name': 'NeuZenAI',
  'tagline': 'Where AI meets ambition.',
  'overview': '''CORE OFFERINGS & MASTER VALUE PROPOSITION
NeuZenAI is a leader in Data, Analytics, and Artificial Intelligence, dedicated to transforming how businesses harness data-driven insights for smarter decisions and intelligently automated operations. The company elevates digital transformation journeys for global enterprises by pioneering the future of AI innovation. NeuZenAI delivers custom ML models, intelligent platforms, and end-to-end product engineering.

PRODUCTS: LENS AI (recruiting), NVISION AI (computer vision), SPENZO (marketing mix modeling), SWASS AI (medical imaging).

TARGET AUDIENCE: global enterprises in E-commerce, Energy, Financial Services, Healthcare, Media, Transportation, Communications, Telecommunications.''',
  'brand_tone': ['Professional', 'Expert', 'Ambitious', 'Forward-thinking', 'Empowering', 'Collaborative', 'Trustworthy'],
  'brand_values': ['Innovation First', 'Human-Centered', 'Ethical AI', 'Excellence', 'Strong Focus on ROI'],
  'brand_aesthetic': ['Modern', 'Clean', 'Professional', 'Data-driven', 'Innovative', 'Intelligent', 'Dynamic'],
  'fonts': ['Inter', 'Sans-serif'],
  'colors': {'accent': '#33CC33', 'primary': '#FF5B00', 'secondary': '#000000', 'background': '#FFFFFF'},
  'url': 'neuzenai.com',
  'logo_url': 'https://neuzenai.com/icon.png',
}

print('Loaded Spenzo + NeuZenAI DNA.')

## 3 — Inputs you change between runs

Edit this cell, re-run it, then re-run the rest top-to-bottom.

In [ ]:
# ============================================================
# EDIT THESE BETWEEN RUNS
# ============================================================

CAMPAIGN_BRIEF = '''Announce Spenzo Pulse — our new real-time MMM dashboard that turns weekly reports into instant clarity for growth teams.'''

BRAND_DNA_MODE = 'spenzo'   # 'none' or 'spenzo'

PLATFORMS = ['linkedin', 'twitter']    # any subset of linkedin / twitter / facebook / instagram

ASPECT_RATIO = '1:1'         # '1:1' / '9:16' / '16:9' / '4:5'

IMAGE_MODEL = 'gemini-3.1-flash-image'   # auto-falls back to gemini-2.5-flash-image if 3.1 not available
N_IMAGE_VARIANTS = 5

print(f'Brief ({len(CAMPAIGN_BRIEF.split())} words) | DNA={BRAND_DNA_MODE} | Platforms={PLATFORMS} | Aspect={ASPECT_RATIO}')

## 4 — `_build_user_context` (DNA → prompt block)

In [ ]:
def build_user_context(mode: str):
    '''Returns (user_context, primary_color, dna_attached_flag).'''
    if mode == 'none':
        return '', '#FF4500', 'no'
    if mode == 'spenzo':
        d = SPENZO_DNA
        ctx_type = 'SPECIFIC PRODUCT DNA: Spenzo AI'
        primary = (d.get('colors') or {}).get('primary', '#FF4500')
    elif mode == 'neuzen':
        d = NEUZEN_COMPANY_DNA
        ctx_type = 'GENERAL COMPANY DNA'
        primary = (d.get('colors') or {}).get('primary', '#FF4500')
    else:
        raise ValueError(f'Unknown BRAND_DNA_MODE: {mode}')

    user_context = f'''
    {ctx_type}
    Entity Name: {d.get('product_name') or d.get('company_name', 'N/A')}
    Target Domain: {d.get('url', 'N/A')}
    Tagline: {d.get('tagline', 'N/A')}
    Brand Values: {', '.join(d.get('brand_values', []))}
    Brand Tone: {', '.join(d.get('brand_tone', []))}
    Brand Aesthetic: {', '.join(d.get('brand_aesthetic', []))}
    Fonts: {', '.join(d.get('fonts', []))}
    Colors: {json.dumps(d.get('colors', {}))}
    Overview: {d.get('overview', 'N/A')}
    '''.strip()
    return user_context, primary, 'yes'

user_context, primary_color, dna_attached = build_user_context(BRAND_DNA_MODE)
print(f'DNA attached: {dna_attached}  |  primary_color: {primary_color}')
print('---')
print(user_context[:600] + ('...' if len(user_context) > 600 else ''))

## 5 — Brief Guard (regex, no LLM)

Rejects junk before any token spend.

In [ ]:
BANNED_TERMS = [
  'sex', 'sexual', 'sexy', 'porn', 'xxx', 'nude', 'nsfw', 'fetish', 'escort',
  'kill', 'murder', 'suicide', 'bomb', 'shoot', 'terrorist', 'assault rifle',
  'cocaine', 'heroin', 'meth', 'drug deal', 'cartel',
  'nazi', 'kkk', 'white supremacy',
  'pyramid scheme', 'ponzi', 'money laundering', 'phishing',
  'automatic weapon', 'machine gun',
]
_BANNED_RX = re.compile(r'\b(?:' + '|'.join(re.escape(t) for t in BANNED_TERMS) + r')\b', re.IGNORECASE)
_GENERIC_RX = [re.compile(p, re.IGNORECASE) for p in [
  r'^\s*(?:create|generate|make)\s+(?:a|an|the)?\s*(?:post|ad|content|image|video)\s*\.?\s*$',
  r'^\s*(?:post|write)\s+(?:about\s+)?(?:something|anything)?\s*\.?\s*$',
  r'^\s*(?:test|hello|hi|hey|asdf|lorem ipsum)\s*\.?\s*$',
]]

def validate_brief(brief: str):
    if not brief or not brief.strip():
        return False, 'Empty brief'
    text = brief.strip(); lower = text.lower()
    if _BANNED_RX.search(lower):
        return False, 'Banned keyword'
    words = [w for w in re.split(r'\s+', lower) if w]
    if len(text) < 15 or len(words) < 4:
        return False, f'Too short: {len(words)} words'
    alpha = sum(1 for c in lower if c.isalpha())
    if alpha / max(1, len(lower)) < 0.45:
        return False, 'Low alpha density'
    for rx in _GENERIC_RX:
        if rx.match(text):
            return False, 'Generic placeholder'
    return True, 'ok'

ok, reason = validate_brief(CAMPAIGN_BRIEF)
print(f'Guard: {"PASS" if ok else "REJECT"} ({reason})')
if not ok:
    raise RuntimeError(f'Guard rejected brief: {reason}')

## 6 — Helper: `call_agent` (matches production `_call_agent`)

In [ ]:
def call_agent(name, prompt, model_name='gemini-2.5-flash', temperature=0.7, web_search=False):
    tools = [types.Tool(google_search=types.GoogleSearch())] if web_search else None
    cfg = types.GenerateContentConfig(temperature=temperature, tools=tools)
    res = client.models.generate_content(model=model_name, contents=prompt, config=cfg)
    text = res.text.strip()
    if '```json' in text:
        text = text.split('```json')[1].split('```')[0].strip()
    elif '```' in text:
        text = text.split('```')[1].split('```')[0].strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as e:
        print(f'[{name}] JSON parse failed: {e}')
        return {'error': str(e), 'raw': text}
    # Attach grounding metadata if web_search was used
    if web_search and isinstance(parsed, dict):
        try:
            gm = res.candidates[0].grounding_metadata if res.candidates else None
            if gm:
                srcs = []
                for c in (gm.grounding_chunks or []):
                    if getattr(c, 'web', None):
                        srcs.append({'uri': getattr(c.web, 'uri', None), 'title': getattr(c.web, 'title', None)})
                parsed.setdefault('_grounding', {})
                parsed['_grounding']['sources'] = srcs
                parsed['_grounding']['queries'] = list(gm.web_search_queries or [])
                print(f'[{name}] grounded: {len(srcs)} sources, {len(parsed["_grounding"]["queries"])} queries')
        except Exception as e:
            print(f'[{name}] grounding metadata extract failed: {e}')
    return parsed

## 7 — Stage 1: Refiner Agent

In [ ]:
def refiner_agent(brief, user_context, dna_attached):
    wc = len(brief.split())
    quality = 'empty' if wc == 0 else 'lean' if wc < 20 else 'specific' if wc < 50 else 'professional'
    knowledge_block = user_context if dna_attached == 'yes' else '(no brand knowledge attached — work from brief only)'
    prompt = f'''
You are a Senior Social Media Growth Strategist.

Your job: take a user's raw campaign brief and produce a structured STRATEGIC BRIEF
that downstream agents (research, copywriter, visualist) can execute against without
guessing what the user meant.

═══════════════════════════════════════════════════════════════
PRIMARY SOURCE OF TRUTH = THE USER'S CAMPAIGN BRIEF
═══════════════════════════════════════════════════════════════
The brief is the only authoritative source for WHAT the campaign is about, WHY it
matters, and any specific facts the user included.
DNA is what you reach for when the brief leaves a gap.

═══════════════════════════════════════════════════════════════
SUPPORTING KNOWLEDGE
═══════════════════════════════════════════════════════════════
DNA attached: {dna_attached}

===== BUSINESS DNA + KNOWLEDGE BASE =====
{knowledge_block}
===== END KNOWLEDGE =====

USE FOR: voice/tone, audience inference, sourced facts.
DO NOT USE FOR: changing the topic the user asked about, inserting a product into
a brief that's about something else, inventing metrics.

═══════════════════════════════════════════════════════════════
USER'S RAW CAMPAIGN BRIEF
═══════════════════════════════════════════════════════════════
({wc} words, quality hint: {quality})

"""
{brief}
"""

═══════════════════════════════════════════════════════════════
STEP 0 — VALIDATE
═══════════════════════════════════════════════════════════════
Reject if HARMFUL / GENERIC / NO_MARKETING_UTILITY.
Cross-topic briefs (DNA = marketing tool, brief = general AI news) are ALLOWED.

If REJECTED, return:
{{ "valid": false, "rejection_category": "...", "rejection_message": "..." }}

═══════════════════════════════════════════════════════════════
STEP 1 — PRODUCE THIS LABELLED STRATEGIC BRIEF
═══════════════════════════════════════════════════════════════
USER GOAL, TOPIC, AUDIENCE, KEY MESSAGE, ANGLE, SUPPORTING POINTS
(every point tagged [user brief] / [DNA: field] / [doc: file]), TONE,
SOURCES REFERENCED, VISUAL HINT, CONSTRAINTS, USER INPUT QUALITY, ASSUMPTIONS MADE.

Never fill TOPIC or KEY MESSAGE by invention.
Source-grounding mandatory — fewer well-sourced points > more invented ones.

Return STRICTLY this JSON (no markdown):
{{ "valid": true, "refined_brief": "<full labelled-sections text block with real newlines>" }}
'''
    return call_agent('REFINER', prompt, temperature=0.7)

ref = refiner_agent(CAMPAIGN_BRIEF, user_context, dna_attached)
if not ref.get('valid', True):
    raise RuntimeError(f'Refiner rejected: {ref}')
REFINED_BRIEF = ref.get('refined_brief', '') if isinstance(ref.get('refined_brief'), str) else json.dumps(ref.get('refined_brief'))
print('REFINED BRIEF:\n')
print(REFINED_BRIEF[:2000] + ('\n...' if len(REFINED_BRIEF) > 2000 else ''))

## 8 — Stage 2a: Cultural Calendar (cached per session)

In [ ]:
_CALENDAR_CACHE = {}
def get_cultural_calendar():
    today = datetime.utcnow().strftime('%Y-%m-%d')
    if today in _CALENDAR_CACHE:
        print('[CULTURAL] cache hit')
        return _CALENDAR_CACHE[today]
    tomorrow = (datetime.utcnow() + timedelta(days=1)).strftime('%Y-%m-%d')
    prompt = f'''
You have access to google_search. Find ONLY major nation-wide cultural moments today/tomorrow
that a mainstream marketer would acknowledge.

Today (UTC): {today}
Tomorrow (UTC): {tomorrow}

INCLUDE for India: national gazetted holidays, major nationally-recognised festivals (Diwali, Holi, Eid, Christmas, Republic Day, Independence Day, Gandhi Jayanti).
INCLUDE for USA: federal holidays + top-tier mainstream marketing days (Valentine's, Mother's, Halloween, Thanksgiving, Black Friday, Cyber Monday).

EXCLUDE: single-state holidays, regional festivals outside their region, UN/WHO observance days, "National X Day" novelties, minor religious observances, niche heritage months.

Return STRICTLY this JSON:
{{
  "today_date": "{today}",
  "tomorrow_date": "{tomorrow}",
  "india": {{ "today": [{{"name":"...","type":"national_holiday|major_festival","note":"..."}}], "tomorrow": [...] }},
  "usa":   {{ "today": [...], "tomorrow": [...] }}
}}
'''
    res = call_agent('CULTURAL_CALENDAR', prompt, temperature=0.2, web_search=True)
    if isinstance(res, dict) and not res.get('error'):
        _CALENDAR_CACHE[today] = res
    return res

CULTURAL = get_cultural_calendar()
print(json.dumps({k: v for k, v in CULTURAL.items() if k != '_grounding'}, indent=2)[:1500])

## 9 — Stage 2b: Researcher (always grounded)

In [ ]:
def format_calendar_for_prompt(cal):
    if not cal or cal.get('error'):
        return '(not available)'
    def _fmt(events):
        if not events: return '  (none notable)'
        return '\n'.join(f'  - {e.get("name","?")} ({e.get("type","?")}): {e.get("note","")}'.rstrip(': ') for e in events)
    return (f"Today ({cal.get('today_date','?')}):\n"
            f"  India:\n{_fmt(cal.get('india',{}).get('today',[]))}\n"
            f"  USA:\n{_fmt(cal.get('usa',{}).get('today',[]))}\n"
            f"Tomorrow ({cal.get('tomorrow_date','?')}):\n"
            f"  India:\n{_fmt(cal.get('india',{}).get('tomorrow',[]))}\n"
            f"  USA:\n{_fmt(cal.get('usa',{}).get('tomorrow',[]))}")

def researcher_agent(refined_brief, user_context, cultural_calendar):
    today = datetime.utcnow().strftime('%Y-%m-%d')
    seven = (datetime.utcnow() - timedelta(days=7)).strftime('%Y-%m-%d')
    cal_block = format_calendar_for_prompt(cultural_calendar)
    prompt = f'''
You are a Research Analyst feeding a downstream Content Agent.
Today's date is {today}.
You HAVE access to Google Search. Ground EVERY claim in real, recent sources.

REFINED BRIEF:
{refined_brief}

USER / BRAND CONTEXT:
{user_context}

LIVE CULTURAL CALENDAR (already fetched):
{cal_block}

═══════════════════════════════════════════════════════════════
PART 1 — MAIN RESEARCH (intent-driven)
═══════════════════════════════════════════════════════════════
Decide based on the brief:
A. TOPIC brief → search topic news last 7 days, primary sources, don't pivot to brand promotion.
B. BRAND brief (DNA attached + brief about own product) → search product context + COMPETITOR news in same category. Populate competitor_news.
C. HYBRID → combine A and B.

═══════════════════════════════════════════════════════════════
PART 2 — AUXILIARY RESEARCH (always, every call)
═══════════════════════════════════════════════════════════════
AUX-1 Cultural calendar (already fetched).
AUX-2 Trending topics in user industry/audience space — emit 3-7.
AUX-3 Trending hashtags — emit 3-10 ("#WithHash").
AUX-4 Trending keywords — emit 3-10.

═══════════════════════════════════════════════════════════════
TIME WINDOW — honor brief
═══════════════════════════════════════════════════════════════
Brief says "today" → only {today}. "yesterday" → 48h. "this week"/none → ≥ {seven}. "this month" → 30d.

═══════════════════════════════════════════════════════════════
LITERAL-QUERY RULE
═══════════════════════════════════════════════════════════════
If the brief contains a specific entity / product name / person / event, pass that string
VERBATIM as one of your google_search queries BEFORE any paraphrased queries.

═══════════════════════════════════════════════════════════════
CITATION RULES
═══════════════════════════════════════════════════════════════
Every concrete fact in trending_context/target_audience/problem_solving_opportunity/company_product_analysis/competitor_news
carries inline [src:N] marker pointing to sources[]. Trending hashtags/keywords/topics must be grounded.

═══════════════════════════════════════════════════════════════
FESTIVAL ALERT
═══════════════════════════════════════════════════════════════
Fire festival_alerts[] when calendar shows a major nation-wide festival today/tomorrow AND brief
didn't mention it.

Return STRICTLY this JSON:
{{
  "target_audience": "...",
  "trending_context": "...",
  "problem_solving_opportunity": "...",
  "company_product_analysis": "...",
  "angles_to_test": ["...", "...", "..."],
  "do_not_claim": ["..."],
  "trending_topics": ["..."],
  "trending_hashtags": ["#One"],
  "trending_keywords": ["..."],
  "competitor_news": [{{"competitor": "...", "headline": "...", "src": 1, "published": "YYYY-MM-DD"}}],
  "festival_alerts": [{{"country": "india|usa", "festival_name": "...", "when": "today|tomorrow", "date": "YYYY-MM-DD", "type": "...", "mentioned_in_brief": false, "suggested_angle": "..."}}],
  "grounding_confidence": "grounded|partial|speculative",
  "sources": [{{"id": 1, "url": "...", "title": "...", "published": "YYYY-MM-DD", "publisher": "..."}}]
}}
'''
    return call_agent('RESEARCHER', prompt, temperature=0.3, web_search=True)

RESEARCH = researcher_agent(REFINED_BRIEF, user_context, CULTURAL)
print('TRENDING TOPICS:', RESEARCH.get('trending_topics'))
print('TRENDING HASHTAGS:', RESEARCH.get('trending_hashtags'))
print('TRENDING KEYWORDS:', RESEARCH.get('trending_keywords'))
print('ANGLES TO TEST:', RESEARCH.get('angles_to_test'))
print('DO NOT CLAIM:', RESEARCH.get('do_not_claim'))
print('FESTIVAL ALERTS:', RESEARCH.get('festival_alerts'))
print('GROUNDING CONFIDENCE:', RESEARCH.get('grounding_confidence'))
print('SOURCES:', json.dumps(RESEARCH.get('sources', [])[:3], indent=2))

## 10 — Stage 3: Copywriter (v3 free-style)

In [ ]:
CHAR_CAP = {'twitter': 270, 'linkedin': 2800, 'facebook': 2200, 'instagram': 2100}
HASHTAG_CAP = {'twitter': 2, 'linkedin': 5, 'facebook': 3, 'instagram': 15}

def copywriter_agent(refined_brief, research, platforms, user_context, cultural_calendar):
    plats_str = ', '.join(platforms)
    has_fest = 'yes' if (research or {}).get('festival_alerts') else 'no'
    cal_block = format_calendar_for_prompt(cultural_calendar)
    prompt = f'''
You are a senior social-media copywriter. Goal: maximize REACH, FOLLOWERS, COMMENTS, SAVES.
Not shares. Real engagement that compounds.

Full creative freedom over voice, structure, hook, CTA, visual format. Pick what fits.

═══════════════════════════════════════════════════════════════
INPUTS
═══════════════════════════════════════════════════════════════
BRAND PROFILE:
{user_context}

REFINED BRIEF (primary source of truth):
{refined_brief}

⚠️ THE REFINED BRIEF IS A STRATEGY DOCUMENT, NOT POST COPY.
Labelled section headers (USER GOAL, TOPIC, AUDIENCE, KEY MESSAGE, ANGLE, SUPPORTING POINTS,
TONE, SOURCES REFERENCED, VISUAL HINT, CONSTRAINTS, USER INPUT QUALITY, ASSUMPTIONS MADE)
are instructions FOR YOU. They are NEVER content to copy into the post.
The user must never see the literal strings "Visual hint:", "Audience:", "Key message:",
"Tone:", etc. in the generated post text.

RESEARCH REPORT:
{json.dumps(research)}

CULTURAL CALENDAR:
{cal_block}

PLATFORMS: {plats_str}
Festival alert active: {has_fest}

═══════════════════════════════════════════════════════════════
HARD RULES
═══════════════════════════════════════════════════════════════
1. ANTI-HALLUCINATION — respect research.do_not_claim. Numbers/dates/product names must trace to brief / DNA / doc / research.sources. No tag = drop.
2. CHAR CAPS (safety-buffered):
   LinkedIn ≤ 2800 | Twitter/X ≤ 270 | Facebook ≤ 2200 | Instagram ≤ 2100
3. HASHTAG CAPS: LinkedIn 0-5 | X 0-2 | FB 0-3 | IG 8-15.
4. NO SHARE-BAIT: "share this", "tag someone who", "RT if you agree", "send to your team".
5. DISTINCT ANGLES — each variant maps to a different research.angles_to_test entry.
6. BRAND VOICE — match brand_tone + brand_values. No press-release "[Brand] is the leader in…". First-person "We're [verb]…" fine.
7. NO BUZZWORDS: empower, democratize, leverage(verb), unlock, transform your workflow, accelerate deployment, unify fragmented, seamlessly, end-to-end, next-gen, best-in-class, streamline, drive efficiency, revolutionize, game-changer, paradigm shift.
8. NO LAZY CTA: "Comment below.", "Let us know in the comments.", "We want to hear your vision.", "Share your thoughts.", "What are your thoughts?"

═══════════════════════════════════════════════════════════════
SOFT GUIDANCE
═══════════════════════════════════════════════════════════════
HOOK MENU: milestone announcement / time-stamped news / open audience question / stat+claim / bold POV / human story / pain-then-solve / product reveal+emoji / punchy contrast.
CTA MENU: arrow+URL / direct verb / open question / soft reply prompt / personal close / thread continuation (X only).
FORMATTING use: single-line paragraphs, emoji bullets (✅⚡🔩🧠1️⃣), numbered lists, pull-quotes, Unicode bold 𝐀𝐁𝐂 only when DNA tone signals tactical/marketing voice.
FORMATTING avoid: arrow bullets ↳, "Click the link in bio" except IG, press-release voice, walls of text.
PLATFORM NATIVE:
  LinkedIn: line-break paragraphs, hashtags final line if any, sweet spot 500-1500.
  X: punchy < 400, `↓` for threads, @-mentions for partner amplification.
  FB: hook in first 480 chars, conversational, photo-paired.
  IG: first 125 chars carry hook, visual rhythm via line breaks + decorative emoji, hashtag wall at end.

═══════════════════════════════════════════════════════════════
VARIANT STRUCTURE
═══════════════════════════════════════════════════════════════
For each platform in {plats_str}, produce 3 variants tied to DIFFERENT entries in research.angles_to_test:
- viral_reach: visibility-flavored, saves+follows, broad-grasp angle.
- high_interaction: comment-driven, genuine replies. CTA invites real reply (NOT "comment A or B").
- follower_growth: authority/depth, promise of more value, follow+save.

FESTIVAL VARIANT: if festival alert active = "yes", ALSO emit `festival_variant` per platform — short voice-faithful festival post using suggested_angle. If "no", OMIT the key entirely.

═══════════════════════════════════════════════════════════════
Return STRICTLY this JSON (no markdown):
═══════════════════════════════════════════════════════════════
{{
  "mode": "product|service|hybrid|topic",
  "mode_reason": "...",
  "recommendation": {{ "best_variant": "viral_reach|high_interaction|follower_growth|festival_variant", "reason": "..." }},
  "content": {{
     "<platform>": {{
       "viral_reach": "...",
       "high_interaction": "...",
       "follower_growth": "...",
       "festival_variant": "..."
     }}
  }}
}}
OMIT festival_variant when festival alert = "no". Do NOT omit any platform.
'''
    return call_agent('COPYWRITER', prompt, temperature=0.7)

def enforce_caps(content_dict):
    out = {}
    for plat, variants in content_dict.items():
        plat_l = plat.lower()
        char_max = CHAR_CAP.get(plat_l, 2200)
        ht_max = HASHTAG_CAP.get(plat_l, 5)
        cleaned = {}
        for k, v in variants.items():
            if not isinstance(v, str):
                cleaned[k] = v; continue
            # Cap hashtags from tail
            tags = re.findall(r'#\w+', v)
            if len(tags) > ht_max:
                excess = tags[ht_max:]
                for t in excess[::-1]:
                    v = re.sub(r'\s*' + re.escape(t) + r'\b', '', v, count=1)
                v = v.rstrip()
            # Truncate to char cap at word boundary
            if len(v) > char_max:
                cut = v[:char_max].rsplit(' ', 1)[0]
                v = cut + '…'
            cleaned[k] = v
        out[plat_l] = cleaned
    return out

COPY = copywriter_agent(REFINED_BRIEF, RESEARCH, PLATFORMS, user_context, CULTURAL)
if 'content' in COPY:
    COPY['content'] = enforce_caps(COPY['content'])

print('MODE:', COPY.get('mode'), '|', COPY.get('mode_reason'))
print('RECOMMENDATION:', COPY.get('recommendation'))
print()
for plat, variants in (COPY.get('content') or {}).items():
    print(f'═══ {plat.upper()} ═══')
    for vk, vt in variants.items():
        print(f'\n--- {vk} ({len(vt)} chars) ---')
        print(vt)
    print()

## 11 — Stage 4: Image Agent v4

5 distinct variants from the AI-recommended copy + brief. No templates, no overlays.
Auto-falls back to `gemini-2.5-flash-image` if `gemini-3.1-flash-image` is unavailable.

In [ ]:
FALLBACK_IMAGE_MODEL = 'gemini-2.5-flash-image'

VISUAL_MOODS = [
  ('photographic_lifestyle',
   'Photorealistic editorial photography. Real people in a real setting that embodies the post. Cinematic lighting, shallow depth of field, magazine-quality composition.'),
  ('product_ui_closeup',
   'Hyper-detailed product UI or device close-up. Modern minimal interface design, crisp edges, bright clean studio lighting. Shows the product doing what the post describes.'),
  ('abstract_metaphor',
   'Abstract 3D illustration capturing the post\'s core idea as a visual metaphor. Soft volumetric lighting, smooth surfaces, gradient backgrounds. Editorial style.'),
  ('data_visualisation',
   'Clean data-driven editorial illustration. Charts, graphs, dashboards, or schematic diagrams rendered in a polished magazine style. No legible labels — visual feel only.'),
  ('flat_vector_illustration',
   'Modern flat vector illustration. Bold colours, geometric shapes, friendly editorial style similar to a top tech-company blog hero illustration.'),
]

ASPECT_HINTS = {
  '1:1': 'Square frame, 1024×1024 pixels.',
  '9:16': 'Vertical portrait frame, 1080×1920 pixels (Stories / Reels).',
  '16:9': 'Wide landscape frame, 1920×1080 pixels.',
  '4:5': 'Tall portrait frame, 1080×1350 pixels (LinkedIn / Instagram engagement-optimal).',
  '3:4': 'Portrait frame, 1080×1440 pixels.',
  '2:3': 'Photo portrait, 1080×1620 pixels.',
}

def get_recommended_copy_per_platform(copy_data):
    best = (copy_data.get('recommendation') or {}).get('best_variant') or 'viral_reach'
    out = {}
    for plat, variants in (copy_data.get('content') or {}).items():
        if not isinstance(variants, dict): continue
        v = variants.get(best) or next((x for x in variants.values() if isinstance(x, str) and x.strip()), None)
        if v: out[plat] = v
    return out

def build_image_prompt(refined_brief, copy_per_platform, primary_color, aspect_ratio, idx, total):
    mood_key, mood_desc = VISUAL_MOODS[idx % len(VISUAL_MOODS)]
    aspect_hint = ASPECT_HINTS.get(aspect_ratio, ASPECT_HINTS['1:1'])
    copy_block = '\n\n'.join(f'--- {p.upper()} ---\n{c}' for p, c in copy_per_platform.items()) or '(no copy provided)'
    return f'''
Create a single high-end social-media post image, ready to publish AS-IS.

═══════════════════════════════════════════════════════════════
CAMPAIGN BRIEF (the strategic context, NOT to be rendered as text in the image)
═══════════════════════════════════════════════════════════════
{refined_brief}

═══════════════════════════════════════════════════════════════
AI-RECOMMENDED COPY THAT WILL ACCOMPANY THIS IMAGE
(the image must visually deliver what these captions promise — the user is going to
publish the caption text alongside this image on each platform)
═══════════════════════════════════════════════════════════════
{copy_block}

═══════════════════════════════════════════════════════════════
VARIANT BRIEF (variant {idx + 1} of {total}: {mood_key})
═══════════════════════════════════════════════════════════════
{mood_desc}

═══════════════════════════════════════════════════════════════
IMAGE SPEC
═══════════════════════════════════════════════════════════════
• Aspect ratio: {aspect_ratio}. {aspect_hint}
• Brand accent colour to weave in subtly: {primary_color}
• 8K, crisp edges, premium publication-grade quality.
• Composition must read at thumbnail size (60% of social engagement is on mobile).

═══════════════════════════════════════════════════════════════
ABSOLUTE PROHIBITIONS — DO NOT RENDER
═══════════════════════════════════════════════════════════════
• No headline text, no caption, no body copy of any kind
• No alphabet letters even as decorative pattern, in any language
• No numbers, digits, percentages, currency symbols, dates
• No corporate logos, brand marks, watermarks, signatures
• No UI mock chrome with readable button labels or menu items
• No street signs, book covers, screen displays with readable text
• No labelled charts (the chart can exist, axis labels cannot)
• No human hands holding signs, papers, devices with text visible

The caption goes OUTSIDE the image — the image must deliver the message visually only.
'''.strip()

def generate_one_variant(prompt, idx, primary_model):
    image_bytes = None
    used_model = primary_model
    try:
        try:
            stream = client.models.generate_content_stream(
                model=primary_model, contents=[prompt],
                config=types.GenerateContentConfig(response_modalities=['IMAGE', 'TEXT']))
            for chunk in stream:
                if chunk.parts:
                    for part in chunk.parts:
                        if part.inline_data:
                            image_bytes = part.inline_data.data; break
                if image_bytes: break
        except Exception as primary_err:
            print(f'[image_v4] variant {idx+1}: primary model {primary_model!r} failed ({primary_err}); fallback {FALLBACK_IMAGE_MODEL!r}')
            used_model = FALLBACK_IMAGE_MODEL
            stream = client.models.generate_content_stream(
                model=FALLBACK_IMAGE_MODEL, contents=[prompt],
                config=types.GenerateContentConfig(response_modalities=['IMAGE', 'TEXT']))
            for chunk in stream:
                if chunk.parts:
                    for part in chunk.parts:
                        if part.inline_data:
                            image_bytes = part.inline_data.data; break
                if image_bytes: break
    except Exception as e:
        print(f'[image_v4] variant {idx+1} failed completely: {e}')
        return None
    if not image_bytes:
        print(f'[image_v4] variant {idx+1}: model returned no image')
        return None
    return {'bytes': image_bytes, 'variant_idx': idx, 'model': used_model, 'prompt': prompt}

def generate_image_variants(refined_brief, copy_data, primary_color, aspect_ratio, n, model):
    copy_per_platform = get_recommended_copy_per_platform(copy_data)
    print(f'[image_v4] firing {n} variants (model={model}, aspect={aspect_ratio}, platforms={list(copy_per_platform.keys())})')
    prompts = [build_image_prompt(refined_brief, copy_per_platform, primary_color, aspect_ratio, i, n) for i in range(n)]
    results = [None] * n
    with ThreadPoolExecutor(max_workers=n) as ex:
        futs = {ex.submit(generate_one_variant, prompts[i], i, model): i for i in range(n)}
        for f in as_completed(futs):
            idx = futs[f]
            results[idx] = f.result()
    succ = [r for r in results if r]
    print(f'[image_v4] {len(succ)}/{n} variants succeeded')
    return succ

IMAGES = generate_image_variants(REFINED_BRIEF, COPY, primary_color, ASPECT_RATIO, N_IMAGE_VARIANTS, IMAGE_MODEL)

for r in IMAGES:
    display(Markdown(f"**Variant {r['variant_idx']+1} · {VISUAL_MOODS[r['variant_idx']][0]} · model: `{r['model']}`**"))
    display(IPyImage(data=r['bytes']))

## 12 — End-to-end output snapshot

In [ ]:
print('=' * 60)
print('PIPELINE COMPLETE')
print('=' * 60)
print(f'Brief: {CAMPAIGN_BRIEF[:100]}...')
print(f'DNA mode: {BRAND_DNA_MODE}')
print(f'Platforms: {PLATFORMS}')
print(f'Recommended variant: {(COPY.get("recommendation") or {}).get("best_variant")}')
print(f'Image variants generated: {len(IMAGES)}/{N_IMAGE_VARIANTS}')
print(f'Sources cited: {len(RESEARCH.get("sources", []))}')
print(f'Web queries fired: {len((RESEARCH.get("_grounding") or {}).get("queries", []))}')
print()
print('Tweak the inputs cell + any prompt cell, then re-run from that cell down.')

---

## How to iterate

1. **Change inputs only** → re-run cells 3 → 7 → 8 → 9 → 10 → 11 → 12.
2. **Change a prompt** → edit the function in its cell, re-run from that cell down.
3. **Compare image models** → set `IMAGE_MODEL = 'gemini-2.5-flash-image'` in cell 3, re-run cell 11 only.
4. **Test "None" DNA mode** → set `BRAND_DNA_MODE = 'none'` in cell 3, re-run from cell 4 down.
5. **Test the NeuZenAI company-level DNA** → set `BRAND_DNA_MODE = 'neuzen'`, re-run from cell 4.

Once you're happy with a prompt change here, copy it into:
- `apps/backend/services/ai_service.py` (refiner / researcher / copywriter / cultural calendar)
- `apps/backend/services/image_agent_v4.py` (image agent)

and restart the backend.
